# Program 7: Transfer Learning with Pre-trained MobileNetV2

**Course:** Deep Learning Lab — 7th Semester, CSE (AI & ML)  
**Environment:** Google Colab (Runtime → Change runtime type → GPU recommended for CNN / transfer learning)

This notebook follows Experiment 7 in the uploaded *DL Lab Manual*. Run cells from top to bottom. The program saves generated artifacts in `/content/DL_Lab_Outputs/Program_7`.

**Note:** Training accuracy, test accuracy, loss values and predictions can vary between runs and hardware. The notebook generates the actual output files when executed in Colab.

In [ ]:
# Import the package or class needed in this experiment.
import numpy as np
# Import the package or class needed in this experiment.
import pandas as pd
# Import the package or class needed in this experiment.
import tensorflow as tf
# Import the package or class needed in this experiment.
import matplotlib.pyplot as plt
# Import the package or class needed in this experiment.
from pathlib import Path

# Perform this processing step in the experiment.
tf.keras.utils.set_random_seed(42)
# Create or define the folder used to store program outputs.
OUT = Path("/content/DL_Lab_Outputs/Program_7")
# Create or define the folder used to store program outputs.
OUT.mkdir(parents=True, exist_ok=True)

# CIFAR-10: 10 image classes
# Assign the resulting value to `(x_train, y_train), (x_test, y_test)`.
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
# Assign the resulting value to `y_train, y_test`.
y_train, y_test = y_train.flatten(), y_test.flatten()
# Assign the resulting value to `class_names`.
class_names = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]

# Resize and apply the preprocessing expected by MobileNetV2
# Assign the resulting value to `IMG_SIZE`.
IMG_SIZE = 96
# Assign the resulting value to `AUTOTUNE`.
AUTOTUNE = tf.data.AUTOTUNE
# Define a helper function for the experiment.
def prepare(images, labels, training=False):
    # Assign the resulting value to `images`.
    images = tf.image.resize(images, (IMG_SIZE, IMG_SIZE))
    # Assign the resulting value to `images`.
    images = tf.keras.applications.mobilenet_v2.preprocess_input(images)
    # Assign the resulting value to `ds`.
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    # Control execution using this condition, loop, or context.
    if training:
        # Assign the resulting value to `ds`.
        ds = ds.shuffle(10000, seed=42)
    # Return the result produced by this function.
    return ds.batch(32).prefetch(AUTOTUNE)

# Assign the resulting value to `train_ds`.
train_ds = prepare(x_train, y_train, training=True)
# Assign the resulting value to `test_ds`.
test_ds = prepare(x_test, y_test)

# Assign the resulting value to `base_model`.
base_model = tf.keras.applications.MobileNetV2(
    # Assign the resulting value to `input_shape`.
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet"
# Perform this processing step in the experiment.
)
# Assign the resulting value to `base_model.trainable`.
base_model.trainable = False

# Assign the resulting value to `inputs`.
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
# Assign the resulting value to `x`.
x = base_model(inputs, training=False)
# Assign the resulting value to `x`.
x = tf.keras.layers.GlobalAveragePooling2D()(x)
# Assign the resulting value to `x`.
x = tf.keras.layers.Dense(64, activation="relu")(x)
# Assign the resulting value to `outputs`.
outputs = tf.keras.layers.Dense(10, activation="softmax")(x)
# Assign the resulting value to `model`.
model = tf.keras.Model(inputs, outputs)

# Configure the optimizer, loss function, and evaluation metrics.
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
# Train the model using the prepared training data.
history = model.fit(train_ds, epochs=5, validation_data=test_ds, verbose=1)
# Evaluate the trained model on the test dataset.
loss, accuracy = model.evaluate(test_ds, verbose=0)
# Display the current result or evaluation metric.
print(f"Test Accuracy: {accuracy*100:.2f}%")

# Assign the resulting value to `sample_images`.
sample_images = x_test[:1]
# Assign the resulting value to `sample_input`.
sample_input = tf.keras.applications.mobilenet_v2.preprocess_input(
    # Perform this processing step in the experiment.
    tf.image.resize(sample_images, (IMG_SIZE, IMG_SIZE))
# Perform this processing step in the experiment.
)
# Generate predictions for the supplied input samples.
probabilities = model.predict(sample_input, verbose=0)[0]
# Assign the resulting value to `predicted_class`.
predicted_class = int(np.argmax(probabilities))
# Display the current result or evaluation metric.
print("Actual:", class_names[int(y_test[0])])
# Display the current result or evaluation metric.
print("Predicted:", class_names[predicted_class])

# Save experiment results to an output file.
pd.DataFrame(history.history).to_csv(OUT/"training_history.csv", index=False)
# Create or transform the data used by the experiment.
pd.DataFrame([{"actual":class_names[int(y_test[0])],
               # Perform this processing step in the experiment.
               "predicted":class_names[predicted_class],
               # Save experiment results to an output file.
               "confidence":float(probabilities[predicted_class])}]).to_csv(OUT/"sample_prediction.csv", index=False)
# Create, format, save, or display the visualization.
plt.plot(history.history["accuracy"], label="Training accuracy")
# Create, format, save, or display the visualization.
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
# Create, format, save, or display the visualization.
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("MobileNetV2 Transfer Learning")
# Create, format, save, or display the visualization.
plt.legend(); plt.tight_layout(); plt.savefig(OUT/"accuracy_curve.png", dpi=150); plt.show()
# Save the trained model or reusable objects.
model.save(OUT/"mobilenetv2_cifar10.keras")